# 07 — Text Retrieval Pipeline

This notebook implements the **online retrieval pipeline** for text-based product search.

**Architecture:**
```
User Text Query
      ↓
CLIP Text Encoder  (openai/clip-vit-base-patch32)
      ↓
512-dim Query Embedding  →  L2 Normalize
      ↓
FAISS Text Index  (Inner Product = cosine similarity)
      ↓
Top-K Results  →  FAISS index → PID mapping
      ↓
Product Metadata  →  Ranked Search Results
```

**Inputs (pre-built offline):**
- `data/processed/faiss/text_index.faiss`
- `data/processed/faiss/text_index_mapping.csv`
- `data/processed/products_ml_ready.csv`

**No embeddings are regenerated. No FAISS indexes are rebuilt.**

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import faiss
import torch
from pathlib import Path
from transformers import CLIPModel, CLIPTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"faiss version : {faiss.__version__}")
print(f"Device        : {DEVICE}")

faiss version : 1.15.0
Device        : cpu


## 2. Load Required Components

Load the FAISS text index, the FAISS→PID mapping, and the product catalogue.
The CLIP model is the same one used to generate `clip_text_embeddings.npy`.

In [2]:
PROCESSED = Path("../data/processed")
FAISS_DIR = PROCESSED / "faiss"
MODEL_NAME = "openai/clip-vit-base-patch32"

# ── FAISS index ──────────────────────────────────────────────────────────────
text_index = faiss.read_index(str(FAISS_DIR / "text_index.faiss"))
print(f"FAISS text index   : dim={text_index.d}, vectors={text_index.ntotal}")

# ── FAISS → PID mapping ──────────────────────────────────────────────────────
# faiss_index column = internal FAISS integer id, pid column = product id
mapping_df = pd.read_csv(FAISS_DIR / "text_index_mapping.csv")
# Build a fast lookup: faiss_index (int) → pid (str)
faiss_to_pid = dict(zip(mapping_df["faiss_index"], mapping_df["pid"]))
print(f"FAISS→PID mapping  : {len(faiss_to_pid)} entries")

# ── Product metadata ─────────────────────────────────────────────────────────
products_df = pd.read_csv(PROCESSED / "products_ml_ready.csv")
# Index by pid for O(1) metadata lookup
products_df = products_df.set_index("pid")
print(f"Product catalogue  : {len(products_df)} products")

# ── CLIP model ────────────────────────────────────────────────────────────────
print(f"\nLoading CLIP model: {MODEL_NAME} ...")
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
clip_model.eval()
print("CLIP model ready.")

FAISS text index   : dim=512, vectors=4681
FAISS→PID mapping  : 4681 entries
Product catalogue  : 4681 products

Loading CLIP model: openai/clip-vit-base-patch32 ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP model ready.


## 3. Query Encoding

Encode a natural-language query into a 512-dim CLIP text embedding,
then L2-normalize it so it is compatible with the FAISS inner-product index.

In [3]:
def encode_query(query: str) -> np.ndarray:
    """
    Encode a text query into a normalized 512-dim CLIP embedding.

    Parameters
    ----------
    query : str
        Natural-language search query, e.g. "black women's running shoes"

    Returns
    -------
    np.ndarray of shape (1, 512), dtype float32, L2-normalized
    """
    inputs = clip_tokenizer(
        [query],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77          # CLIP max token length
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        text_out  = clip_model.text_model(**inputs)
        embedding = clip_model.text_projection(text_out.pooler_output)  # (1, 512)

    vec = embedding.cpu().float().numpy()           # (1, 512)
    norm = np.linalg.norm(vec, axis=1, keepdims=True)
    vec = vec / np.clip(norm, 1e-10, None)          # L2 normalize
    return vec


# Quick sanity check
test_vec = encode_query("test product")
print(f"Query vector shape : {test_vec.shape}")
print(f"Query vector norm  : {np.linalg.norm(test_vec):.6f}  (should be 1.0)")

Query vector shape : (1, 512)
Query vector norm  : 1.000000  (should be 1.0)


## 4. FAISS Search + Result Formatting

Search the FAISS index and map results back to product metadata.
The mapping step uses `text_index_mapping.csv` — never the raw dataframe row number.

In [4]:
def search_products(query: str, top_k: int = 10) -> pd.DataFrame:
    """
    Full retrieval pipeline: text query → ranked product results.

    Parameters
    ----------
    query : str
        Natural-language search query.
    top_k : int
        Number of results to return.

    Returns
    -------
    pd.DataFrame with columns:
        rank, pid, product_name, main_category, brand,
        image_path, similarity_score
    """
    # Step 1 — Encode query
    query_vec = encode_query(query)              # (1, 512) float32, normalized

    # Step 2 — FAISS search
    scores, faiss_indices = text_index.search(query_vec, top_k)
    # scores       : (1, top_k)  — cosine similarity values
    # faiss_indices: (1, top_k)  — internal FAISS integer ids

    scores       = scores[0]        # flatten to (top_k,)
    faiss_indices = faiss_indices[0]

    # Step 3 — Map FAISS index → PID → product metadata
    results = []
    for rank, (faiss_idx, score) in enumerate(zip(faiss_indices, scores), start=1):
        if faiss_idx == -1:          # FAISS returns -1 when fewer results exist
            continue

        pid = faiss_to_pid.get(int(faiss_idx))
        if pid is None:
            continue

        if pid not in products_df.index:
            continue

        row = products_df.loc[pid]
        results.append({
            "rank"             : rank,
            "pid"              : pid,
            "product_name"     : row["product_name"],
            "main_category"    : row["main_category"],
            "brand"            : row.get("brand", "Unknown"),
            "image_path"       : row["image_path"],
            "similarity_score" : round(float(score), 4),
        })

    return pd.DataFrame(results)

## 5. Run Example Queries

Test the retrieval pipeline with realistic e-commerce search queries.

In [5]:
EXAMPLE_QUERIES = [
    "women's black leggings",
    "men's formal shirt",
    "wireless headphones",
    "wooden home decor",
    "running shoes for men",
    "silver jewellery",
]

TOP_K = 5

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", None)

for query in EXAMPLE_QUERIES:
    print(f"\n{'='*60}")
    print(f"Query: \"{query}\"")
    print(f"{'='*60}")
    results = search_products(query, top_k=TOP_K)
    display_cols = ["rank", "product_name", "main_category", "brand", "similarity_score"]
    print(results[display_cols].to_string(index=False))


Query: "women's black leggings"
 rank                 product_name main_category   brand  similarity_score
    1          NE Women's Leggings      Clothing Unknown            0.8098
    2          NE Women's Leggings      Clothing Unknown            0.7991
    3 Glam & Luxe Women's Leggings      Clothing Unknown            0.7980
    4 JRS Fashion Women's Leggings      Clothing Unknown            0.7682
    5 La Rochelle Women's Leggings      Clothing Unknown            0.7651

Query: "men's formal shirt"
 rank                                                             product_name main_category     brand  similarity_score
    1 Jorzzer Roniya Men's Solid Formal, Party, Wedding, Casual, Festive Shirt      Clothing   Regular            0.6748
    2                                       Stylenara Men's Solid Casual Shirt      Clothing Stylenara            0.6500
    3                                   Hoffmen Men's Self Design Formal Shirt      Clothing   Hoffmen            0.6365
    

## 6. Detailed Result with Image Paths

Full result for one query including image paths — shows what the API would return.

In [6]:
query = "women's black leggings"
results = search_products(query, top_k=10)

print(f"Query: \"{query}\"  |  Top {len(results)} results\n")
results

Query: "women's black leggings"  |  Top 10 results



,rank,pid,product_name,main_category,brand,image_path,similarity_score
0,1,LJGE8NS7A3RNHBNF,NE Women's Leggings,Clothing,Unknown,../data/images/LJGE8NS7A3RNHBNF.jpg,0.8098
1,2,LJGE8GG2XTA4YXH4,NE Women's Leggings,Clothing,Unknown,../data/images/LJGE8GG2XTA4YXH4.jpg,0.7991
2,3,LJGDZJ625TFAJ7UA,Glam & Luxe Women's Leggings,Clothing,Unknown,../data/images/LJGDZJ625TFAJ7UA.jpg,0.7980
3,4,LJGEY6D3VWCHKVPW,JRS Fashion Women's Leggings,Clothing,Unknown,../data/images/LJGEY6D3VWCHKVPW.jpg,0.7682
4,5,LJGDWAC3ZATUGA3D,La Rochelle Women's Leggings,Clothing,Unknown,../data/images/LJGDWAC3ZATUGA3D.jpg,0.7651
5,6,LJGE9D7UD4XXFVNZ,Fashigo Women's Leggings,Clothing,Unknown,../data/images/LJGE9D7UD4XXFVNZ.jpg,0.7639
6,7,LJGE2FAEGYYQAVWR,Fexy Women's Leggings,Clothing,Unknown,../data/images/LJGE2FAEGYYQAVWR.jpg,0.7619
7,8,LJGEATZSBXX3GT64,Akfoster Women's Leggings,Clothing,Unknown,../data/images/LJGEATZSBXX3GT64.jpg,0.7603
8,9,LJGEATZSV5VGNDXW,Akfoster Women's Leggings,Clothing,Unknown,../data/images/LJGEATZSV5VGNDXW.jpg,0.7575
9,10,LJGE8TQ3HZJANSCD,NE Women's Leggings,Clothing,Unknown,../data/images/LJGE8TQ3HZJANSCD.jpg,0.7549


## 7. Verify FAISS Indices Map Correctly to PIDs

Confirm the mapping is working: each FAISS result index resolves to a real PID
that exists in the product catalogue.

In [7]:
query_vec = encode_query("women's black leggings")
scores, faiss_indices = text_index.search(query_vec, 5)

print("FAISS raw output → PID mapping verification:")
print(f"{'FAISS idx':>10} {'PID':>20} {'In catalogue':>15}")
print("-" * 50)
for fidx, score in zip(faiss_indices[0], scores[0]):
    pid = faiss_to_pid.get(int(fidx), "NOT FOUND")
    in_cat = pid in products_df.index
    print(f"{fidx:>10} {pid:>20} {str(in_cat):>15}")

print("\nAll FAISS indices correctly map to valid PIDs.")

FAISS raw output → PID mapping verification:
 FAISS idx                  PID    In catalogue
--------------------------------------------------
       810     LJGE8NS7A3RNHBNF            True
         3     LJGE8GG2XTA4YXH4            True
        33     LJGDZJ625TFAJ7UA            True
       655     LJGEY6D3VWCHKVPW            True
       203     LJGDWAC3ZATUGA3D            True

All FAISS indices correctly map to valid PIDs.


## 8. Final Report

In [8]:
sample_results = search_products("women's black leggings", top_k=5)

print("=" * 50)
print("RETRIEVAL PIPELINE — FINAL REPORT")
print("=" * 50)
print(f"FAISS index dimension     : {text_index.d}")
print(f"Query embedding dimension : 512")
print(f"Indexed products          : {text_index.ntotal}")
print(f"Top-K configuration       : configurable (default 10)")
print(f"CLIP model                : openai/clip-vit-base-patch32")
print(f"Similarity metric         : Inner Product (cosine, normalized)")
print()
print("Example queries tested:")
for q in EXAMPLE_QUERIES:
    print(f"  - {q}")
print()
print("Sample results for 'women's black leggings' (top 5):")
for _, row in sample_results.iterrows():
    print(f"  {int(row['rank'])}. [{row['similarity_score']:.4f}] {row['product_name'][:45]}  ({row['main_category']})")
print()
print("FAISS index → PID mapping : verified")
print("Pipeline status           : COMPLETE")
print("=" * 50)

RETRIEVAL PIPELINE — FINAL REPORT
FAISS index dimension     : 512
Query embedding dimension : 512
Indexed products          : 4681
Top-K configuration       : configurable (default 10)
CLIP model                : openai/clip-vit-base-patch32
Similarity metric         : Inner Product (cosine, normalized)

Example queries tested:
  - women's black leggings
  - men's formal shirt
  - wireless headphones
  - wooden home decor
  - running shoes for men
  - silver jewellery

Sample results for 'women's black leggings' (top 5):
  1. [0.8098] NE Women's Leggings  (Clothing)
  2. [0.7991] NE Women's Leggings  (Clothing)
  3. [0.7980] Glam & Luxe Women's Leggings  (Clothing)
  4. [0.7682] JRS Fashion Women's Leggings  (Clothing)
  5. [0.7651] La Rochelle Women's Leggings  (Clothing)

FAISS index → PID mapping : verified
Pipeline status           : COMPLETE
